# C9-dimensionality-reduction — Practice p18 — Solution

In [1]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
n = 600
t = np.sort(rng.uniform(-1.5 * np.pi, 1.5 * np.pi, n))
y = rng.uniform(0.0, 6.0, n)
X3 = np.column_stack([np.sin(t), y, np.sign(t) * (np.cos(t) - 1.0)]) \
     + rng.normal(0.0, 0.03, (n, 3))
Xc = X3 - X3.mean(axis=0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
P2 = Xc @ Vt[:2].T
UV = np.column_stack([t, y])
P1 = Xc @ Vt[:1].T

def knn_preservation(A_orig, A_view, k):
    sq_orig = (A_orig * A_orig).sum(axis=1)
    D2_orig = np.maximum(sq_orig[:, None] + sq_orig[None, :] - 2.0 * (A_orig @ A_orig.T), 0.0)
    sq_view = (A_view * A_view).sum(axis=1)
    D2_view = np.maximum(sq_view[:, None] + sq_view[None, :] - 2.0 * (A_view @ A_view.T), 0.0)
    np.fill_diagonal(D2_orig, np.inf)
    np.fill_diagonal(D2_view, np.inf)
    nb_orig = np.argsort(D2_orig, axis=1)[:, :k]
    nb_view = np.argsort(D2_view, axis=1)[:, :k]
    fractions = np.empty(A_orig.shape[0], dtype=np.float64)
    for i in range(A_orig.shape[0]):
        fractions[i] = np.intersect1d(nb_orig[i], nb_view[i]).size / k
    return float(fractions.mean())

pres = {
    "p2": float(knn_preservation(X3, P2, 10)),
    "p1": float(knn_preservation(X3, P1, 10)),
    "uv": float(knn_preservation(X3, UV, 10)),
}

D_orig = np.sqrt(((X3[:, None, :] - X3[None, :, :]) ** 2).sum(axis=2))
D_p2 = np.sqrt(((P2[:, None, :] - P2[None, :, :]) ** 2).sum(axis=2))
D_p1 = np.sqrt(((P1[:, None, :] - P1[None, :, :]) ** 2).sum(axis=2))
D_uv = np.sqrt(((UV[:, None, :] - UV[None, :, :]) ** 2).sum(axis=2))
iu = np.triu_indices(n, 1)
stretch = {
    "p2": float(np.max(D_p2[iu] / D_orig[iu])),
    "p1": float(np.max(D_p1[iu] / D_orig[iu])),
    "uv": float(np.max(D_uv[iu] / D_orig[iu])),
}

far_on_ribbon = np.abs(t[:, None] - t[None, :]) > 2.0
masked_view = np.where(far_on_ribbon, D_p2, np.inf)
pair = np.unravel_index(np.argmin(masked_view), masked_view.shape)
fn_view_dist = float(D_p2[pair])
fn_true_dist = float(D_orig[pair])

print("preservation:", pres)
print("stretch:", stretch)
print("false-neighbor pair / view / true:", pair, fn_view_dist, fn_true_dist)

preservation: {'p2': 0.5876666666666666, 'p1': 0.13033333333333333, 'uv': 0.9321666666666666}
stretch: {'p2': 0.9999999999998912, 'p1': 0.99999839031772, 'uv': 4.303199004892209}
false-neighbor pair / view / true: (np.int64(27), np.int64(177)) 0.023111768437960006 1.9531639139520762


| Question | Best view | Evidence (computed numbers) |
|---|---|---|
| “Are these two points genuine neighbors?” | Unrolled (`uv`) | Preservation is $0.9322$, versus $0.5877$ for `p2`; it wins the local metric. |
| “How far apart are these two regions, really?” | `p2` (then original-space distances for the number) | `p2` stretch is $1.0000$, so it never exaggerates, but a false neighbor is $0.0231$ apart in the view and $1.9532$ apart in 3-D; `uv` stretches some pairs by $4.3032$. |
| “Is a 1-D summary usable for either question?” | No | `p1` preserves only $0.1303$ of neighbor memberships; although its stretch is at most $1$, that only prevents exaggeration and does not stop severe collapse. |

### Answer check

In [2]:
assert P2.shape == (600, 2) and P1.shape == (600, 1) and UV.shape == (600, 2)
assert np.isclose(pres["p2"], 0.5876666666666666, atol=1e-12, rtol=0)
assert np.isclose(pres["p1"], 0.13033333333333333, atol=1e-12, rtol=0)
assert np.isclose(pres["uv"], 0.9321666666666666, atol=1e-12, rtol=0)
assert np.isclose(stretch["p2"], 1.0, atol=1e-12, rtol=0)
assert np.isclose(stretch["p1"], 0.9999983903177203, atol=1e-12, rtol=0)
assert np.isclose(stretch["uv"], 4.303199004892209, atol=1e-12, rtol=0)
assert np.isclose(fn_view_dist, 0.02311176843796016, atol=1e-12, rtol=0)
assert np.isclose(fn_true_dist, 1.9531639139520762, atol=1e-12, rtol=0)